<a href="https://colab.research.google.com/github/prithwis/GeoSim/blob/main/gsWorldFactCollector_NewsLines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![alt text](https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/images/CentaurBanner2.png)<br>


<hr>

[Prithwis Mukerjee](http://www.linkedin.com/in/prithwis)<br>

In [1]:
!pip -q install pymupdf
!wget -O gsLLM.py -q https://raw.githubusercontent.com/prithwis/GeoSim/refs/heads/main/gsUtils/gsLLM.py
!wget -O gsWorldFacts.py -q https://raw.githubusercontent.com/prithwis/GeoSim/refs/heads/main/gsUtils/gsWorldFacts.py                  # all

from gsLLM import *
from gsWorldFacts import *

from pathlib import Path
import time

In [3]:
import importlib
import gsLLM, gsWorldFacts
importlib.reload(gsLLM)
importlib.reload(gsWorldFacts)

<module 'gsWorldFacts' from '/content/gsWorldFacts.py'>

In [5]:
# ------------------------------------------------
# Use OpenAI
# ------------------------------------------------
provider = "openai"

#model = "gpt-5.6-luna"
model = "gpt-6-luna"
#model = "gpt-6.1-sol"
#model = "gpt-6-astra"

# ------------------------------------------------
# Use Claude
# ------------------------------------------------
#!pip install -q anthropic

#provider = "claude"
#model = "claude-haiku-5-5"
#model = "claude-sonnet-5-5"

# ------------------------------------------------
# Authenticate wit LLM provider using API keys stored as Colab Secret
# ------------------------------------------------
client = authenticateLLM(provider=provider)

from google.colab import drive
drive.mount('/content/drive')

OPENAI | User: Calcutta | Email: prithwis@yantrajaal.com | ID: user-sNAljKNmBf2igtAV2kUuqhoF
Mounted at /content/drive


In [7]:
# ------------------------------------------------
# Locate and load all domain informatio
# ------------------------------------------------

DOMAIN = 'Silicon'
#DOMAIN = 'MOS2'


GeoSim_HOME = "/content/drive/MyDrive/Prithwis/GeoSim/"
Domain_HOME = "/content/drive/MyDrive/Prithwis/GeoSim/"+DOMAIN
#print(GeoSim_HOME)

WORLD_FACTS = Path(Domain_HOME) / "WorldFacts/"

ROLE_DIR = Path(GeoSim_HOME) / "Roles"

!ls -l "{WORLD_FACTS}"

total 16
drwx------ 2 root root 4096 Nov 19 07:16 Markdown
drwx------ 2 root root 4096 Nov 21 10:39 PDF_Original_1
drwx------ 2 root root 4096 Nov 21 10:39 PDF_Original_2
-rw------- 1 root root 3421 Nov 19 07:34 Silicon_Stack_Source_Manifest.md


## Live News Feed Collection and Storage
Currently only from MoneyControl

In [8]:
_mc_links = get_mc_links("https://www.moneycontrol.com/world/", "/world/")

/world/ links located : 60


In [9]:
stories = get_stories(_mc_links)

Processed 10/60 | Valid stories: 9
Processed 20/60 | Valid stories: 18
Processed 30/60 | Valid stories: 25
Processed 40/60 | Valid stories: 30
Processed 50/60 | Valid stories: 35
Processed 60/60 | Valid stories: 42
----------------------------------------
Links examined : 60
Valid stories  : 42


In [15]:
import json


relevanceFile = f"{Domain_HOME}/NewsConfig/NewsRelevance.json"

with open(relevanceFile, "r", encoding="utf-8") as f:
    relevance = json.load(f)

print(relevanceFile)
print(Path(relevanceFile).exists())

/content/drive/MyDrive/Prithwis/GeoSim/Silicon/NewsConfig/NewsRelevance.json
True


In [16]:
# ------------------------------------------------------------------------
# Score stories using scenario-specific relevance criteria
# ------------------------------------------------------------------------

def scoreStory(article, relevance):

    text = (
        (article.get("title") or "")
        + " "
        + (article.get("content") or "")
    ).lower()

    score = 0

    for category in relevance.values():

        weight = category["weight"]

        for keyword in category["keywords"]:
            if keyword.lower() in text:
                score += weight

    return score

In [17]:
# ------------------------------------------------------------------------
# Score and sort stories by relevance
# ------------------------------------------------------------------------

for s in stories:
    s["score"] = scoreStory(s, relevance)

stories = sorted(
    stories,
    key=lambda x: x["score"],
    reverse=True
)

for i, s in enumerate(stories, 1):
    print(f"{i:3}. [{s['score']:2}] {s['title']}")

  1. [12] US intel finds no evidence of China plot in Nvidia CEO and Alice Huang's affair: Report
  2. [10] Chips, SK Hynix and AI boom: Why South Korea’s ‘divorce of the century’ making headlines
  3. [10] US officials divided over Chinese spying allegations following Jensen Huang's return: Report
  4. [ 9] Fifth Amendment explained: Why Trump doesn't want Nvidia CEO Jensen Huang to invoke it
  5. [ 8] No proof Alice Huang was sent by Beijing to spy on Jensen Huang, US intelligence tells Congress: Report
  6. [ 6] Trump targets 14 economies with fresh tariffs, but Pakistan gets a pass: What explains the exclusion?
  7. [ 6] Trump’s tariff list: India faces 17% duty under trade deal, Vietnam 20%; Check country-wise rates
  8. [ 6] US plans fresh tariffs on India, China and EU after Supreme Court strikes down Trump's earlier levies
  9. [ 5] Alice Huang-linked firms listed $270 million in AI servers in China: Report
 10. [ 5] Microdramas rake in $20 billion as world gloms on to quick hi

In [18]:
for i, s in enumerate(stories, 1):
    print(
        f"{i:3}. "
        f"[{s['score']:3}] "
        f"[{len(s.get('content') or ''):6} chars] "
        f"{s['title']}"
    )

  1. [ 12] [  1446 chars] US intel finds no evidence of China plot in Nvidia CEO and Alice Huang's affair: Report
  2. [ 10] [  1340 chars] Chips, SK Hynix and AI boom: Why South Korea’s ‘divorce of the century’ making headlines
  3. [ 10] [  1611 chars] US officials divided over Chinese spying allegations following Jensen Huang's return: Report
  4. [  9] [  1432 chars] Fifth Amendment explained: Why Trump doesn't want Nvidia CEO Jensen Huang to invoke it
  5. [  8] [  1603 chars] No proof Alice Huang was sent by Beijing to spy on Jensen Huang, US intelligence tells Congress: Report
  6. [  6] [  1696 chars] Trump targets 14 economies with fresh tariffs, but Pakistan gets a pass: What explains the exclusion?
  7. [  6] [  1549 chars] Trump’s tariff list: India faces 17% duty under trade deal, Vietnam 20%; Check country-wise rates
  8. [  6] [  1470 chars] US plans fresh tariffs on India, China and EU after Supreme Court strikes down Trump's earlier levies
  9. [  5] [  1471 chars] Ali

In [19]:
# ------------------------------------------------------------------------
# Inspect top MOS-ranked stories
# ------------------------------------------------------------------------

for i, s in enumerate(stories[:15], 1):

    print("=" * 80)
    print(f"{i}. SCORE: {s['score']}")
    print(s["title"])
    print()
    print(s["content"])
    print()

1. SCORE: 12
US intel finds no evidence of China plot in Nvidia CEO and Alice Huang's affair: Report

The United States has found no evidence that China orchestrated Alice Huang's affair with Jensen Huang, to spy on him, intelligence officials told Congress at a classified briefing a day after the Nvidia founder’s surprise reinstatement as CEO, the Wall Street Journal reported, citing people familiar with the matter. US intelligence officials said Alice Huang was in contact with China’s Ministry of State Security (MSS), which had approached her for information under Chinese law. However, they found no indication that she passed significant intelligence to the agency, according to the WSJ report. In fact, current evidence collected by US intelligence suggests the Chinese woman's relationship with the Nvidia boss may have been amorous, not a coordinated play by Beijing. The briefing, however, did not settle questions surrounding Alice Huang’s role in the alleged diversion of advanced Nvi

In [ ]:
live_file = CreateLiveFeed(stories, 4)

Stories selected : 22
File created     : MOS-MC-LiveFeed_261005_1524_IST.txt


In [ ]:
!ls -l "{WORLDFACTS}/NEWS"

total 71
-rw------- 1 root root 71893 Oct  2 07:14 MOS-MC-LiveFeed_261002_1244_IST.txt


In [ ]:
!rm "{WORLDFACTS}"/NEWS/*.txt

In [ ]:
!ls -l "{WORLDFACTS}"
NEWS_DIR = f"{WORLDFACTS}/NEWS"

!ls -l "{WORLDFACTS}/NEWS"
!ls -l "{WORLDFACTS}/MD"

total 20
drwx------ 2 root root 4096 Oct  5 09:36 MD
drwx------ 2 root root 4096 Oct  5 09:36 MD_V0
drwx------ 2 root root 4096 Oct  5 09:54 NEWS
drwx------ 2 root root 4096 Oct  5 09:34 PDF
drwx------ 2 root root 4096 Oct  5 09:33 PDF_v1
total 0
total 483
-rw------- 1 root root  81138 Oct  5 09:41 01R4732122_261005.md
-rw------- 1 root root 128802 Oct  5 09:45 02184iranspriorities_261005.md
-rw------- 1 root root 189140 Oct  5 09:50 03crsdefensej18_261005.md
-rw------- 1 root root  31604 Oct  5 09:51 04191211IranianMissi_261005.md
-rw------- 1 root root  62143 Oct  5 09:53 0520180913irangccvak_261005.md


In [ ]:
!mv "{live_file}" "{NEWS_DIR}/"

In [ ]:
!ls -l "{MD_DIR}"
!ls -l "{NEWS_DIR}"

ls: cannot access '{MD_DIR}': No such file or directory
total 38
-rw------- 1 root root 38420 Oct  5 09:54 MOS-MC-LiveFeed_261005_1524_IST.txt


In [ ]:
#Check balance
# https://platform.openai.com/usage?utm_source=chatgpt.com